# Task 10: Pairwise rank correlations

Task 9 fixed the comparison group at 663 occupations. Here we measure how similarly AIOE, Frey–Osborne, and Eloundou order those occupations, first with GPT-4 beta and then with human beta. Alpha and gamma are sensitivity checks on exactly the same sample.

- **Spearman rho:** correlation of ranked positions.
- **Kendall tau-b:** agreement in occupation-pair ordering, adjusted for ties.
- Coefficients near +1 indicate similar ordering, near 0 little consistent ordering, and near −1 opposite ordering.
- **Label-source difference:** human coefficient minus GPT-4 coefficient, within the same method, variant, and index pair. This is a signed change, not a change in absolute correlation strength.

Run top to bottom from the repository root or `notebooks/`, using dependencies in `requirements.txt`. No scores are imputed, and the fixed Task 9 sample is not modified. Headline images go in `figures/`; the tidy CSV and sensitivity appendix go in `data/datasets/processed/`.

These are descriptive, unweighted occupation correlations. Exposure is not a prediction of job loss.

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy

project_folder = Path('..') if Path('../data/datasets/processed').is_dir() else Path('.')
processed_folder = project_folder / 'data/datasets/processed'
figures_folder = project_folder / 'figures'
sample_path = processed_folder / 'agreement_analysis_sample.csv'
input_bytes = sample_path.read_bytes()
source_sha256 = hashlib.sha256(input_bytes).hexdigest()
sample = pd.read_csv(sample_path, dtype={'soc2018': 'string'})

variants = ['beta', 'alpha', 'gamma']
label_sources = {'gpt4': 'dv_rating', 'human': 'human_rating'}
methods = ['spearman', 'kendall']
score_to_rank = {
    'aioe': 'rank_aioe', 'fo_prob': 'rank_fo',
    'dv_rating_beta': 'rank_elo_dv', 'human_rating_beta': 'rank_elo_hum',
}
raw_columns = ['aioe', 'fo_prob'] + [
    prefix + '_' + variant
    for prefix in label_sources.values() for variant in variants
]
assert {'soc2018', 'included_in_analysis', *raw_columns, *score_to_rank.values()}.issubset(sample.columns)
assert len(sample) == 663, f'Expected the fixed 663 occupations; found {len(sample)}'
assert sample['soc2018'].notna().all() and sample['soc2018'].is_unique
assert sample['soc2018'].str.fullmatch(r'\d{2}-\d{4}').all()
assert sample['included_in_analysis'].eq(True).all()
assert sample[raw_columns].notna().all().all(), 'Missing scores: do not silently drop occupations'
for column in raw_columns:
    values = pd.to_numeric(sample[column], errors='raise')
    assert values.map(math.isfinite).all(), f'Nonfinite score in {column}'
    assert values.nunique() > 1, f'Constant scores in {column}: correlation undefined'
    if column != 'aioe':
        assert values.between(0, 1).all(), f'Invalid share/probability in {column}'
for score, rank in score_to_rank.items():
    np.testing.assert_allclose(
        sample[rank], sample[score].rank(method='average', ascending=True, pct=True),
        rtol=0, atol=1e-12,
    )

print(f'Fixed sample verified: {len(sample)} unique occupations; all beta/alpha/gamma scores present.')
print(f'Input SHA-256: {source_sha256}')
print(f'pandas {pd.__version__}; SciPy {scipy.__version__}')

Fixed sample verified: 663 unique occupations; all beta/alpha/gamma scores present.
Input SHA-256: 7f0c606895d080d913ed147c8436a60ea61cafeebd1dfe3a9bba907509f782ee
pandas 2.3.3; SciPy 1.13.1


## 1. Calculate the matrices and tidy pairwise results

Each run contains three indices and uses all 663 occupations. Beta uses Task 9's saved ranks. Alpha/gamma ranks are calculated within the same sample with the same average-tie rule. AIOE and Frey–Osborne ranks are unchanged in every run.

The full matrices include both mirrored halves and diagonals. The tidy CSV keeps only the three distinct pairs per run: AIOE–Frey–Osborne, AIOE–Eloundou, and Frey–Osborne–Eloundou. It records the exact raw score variants behind the ranks.

In [2]:
matrices = {}
records = []
for variant in variants:
    for label_source, prefix in label_sources.items():
        eloundou_score = prefix + '_' + variant
        if variant == 'beta':
            eloundou_rank = sample['rank_elo_dv' if label_source == 'gpt4' else 'rank_elo_hum']
        else:
            eloundou_rank = sample[eloundou_score].rank(method='average', ascending=True, pct=True)
        ranks = pd.DataFrame({
            'aioe': sample['rank_aioe'],
            'frey_osborne': sample['rank_fo'],
            'eloundou': eloundou_rank,
        })
        raw_names = {'aioe': 'aioe', 'frey_osborne': 'fo_prob', 'eloundou': eloundou_score}
        assert ranks.index.equals(sample.index)
        assert ranks.notna().all().all()
        assert ranks.gt(0).all().all() and ranks.le(1).all().all()
        counts = ranks.notna().astype('int64').T.dot(ranks.notna().astype('int64'))
        assert counts.eq(663).all().all()

        for method in methods:
            matrix = ranks.corr(method=method)
            assert np.isfinite(matrix.to_numpy()).all()
            np.testing.assert_allclose(matrix, matrix.T, rtol=0, atol=1e-12)
            np.testing.assert_allclose(np.diag(matrix), 1, rtol=0, atol=1e-12)
            assert (np.abs(matrix.to_numpy()) <= 1 + 1e-12).all()
            matrices[(variant, label_source, method)] = matrix
            for index_x, index_y in combinations(ranks.columns, 2):
                records.append({
                    'variant': variant, 'label_source': label_source, 'method': method,
                    'index_x': index_x, 'index_y': index_y,
                    'score_x': raw_names[index_x], 'score_y': raw_names[index_y],
                    'coefficient': matrix.loc[index_x, index_y],
                    'n': int(counts.loc[index_x, index_y]),
                })

results = pd.DataFrame(records)
pair_keys = ['variant', 'method', 'index_x', 'index_y']
paired = results.pivot(index=pair_keys, columns='label_source', values='coefficient')
assert paired.notna().all().all()
paired['human_minus_gpt4'] = paired['human'] - paired['gpt4']
results = results.merge(
    paired[['human_minus_gpt4']].reset_index(), on=pair_keys,
    how='left', validate='many_to_one',
)
assert len(matrices) == 12 and len(results) == 36
assert not results.duplicated(pair_keys + ['label_source']).any()
assert results['n'].eq(663).all()
baseline = results.loc[results['index_y'].eq('frey_osborne')]
assert baseline['human_minus_gpt4'].eq(0).all()
for method in methods:
    assert baseline.loc[baseline['method'].eq(method), 'coefficient'].nunique() == 1

print('Headline beta results (difference = human minus GPT-4):')
print(results.loc[results['variant'].eq('beta')].to_string(index=False))
print(f'Built {len(matrices)} complete matrices and {len(results)} distinct-pair rows.')

Headline beta results (difference = human minus GPT-4):
variant label_source   method      index_x      index_y score_x           score_y  coefficient   n  human_minus_gpt4
   beta         gpt4 spearman         aioe frey_osborne    aioe           fo_prob    -0.367823 663          0.000000
   beta         gpt4 spearman         aioe     eloundou    aioe    dv_rating_beta     0.862805 663          0.004143
   beta         gpt4 spearman frey_osborne     eloundou fo_prob    dv_rating_beta    -0.267008 663         -0.073109
   beta         gpt4  kendall         aioe frey_osborne    aioe           fo_prob    -0.249075 663          0.000000
   beta         gpt4  kendall         aioe     eloundou    aioe    dv_rating_beta     0.663246 663          0.007928
   beta         gpt4  kendall frey_osborne     eloundou fo_prob    dv_rating_beta    -0.182787 663         -0.050605
   beta        human spearman         aioe frey_osborne    aioe           fo_prob    -0.367823 663          0.000000
   beta 

## 2. Check known examples and verify score/rank agreement

A known tied example checks that Kendall uses tau-b. Every result is also compared with SciPy calculated directly on the preserved raw scores. Rank correlations should be the same whether calculated on raw scores or their average percentile ranks, because the ordering and ties are unchanged.

In [3]:
from scipy.stats import spearmanr, kendalltau

example = pd.DataFrame({'x': [1, 1, 2], 'y': [1, 2, 3]})
assert math.isclose(example.corr(method='spearman').loc['x', 'y'], math.sqrt(3) / 2)
assert math.isclose(example.corr(method='kendall').loc['x', 'y'], math.sqrt(2 / 3))
reverse = pd.DataFrame({'x': [1, 2, 3], 'y': [3, 2, 1]})
for method in methods:
    assert math.isclose(reverse.corr(method=method).loc['x', 'y'], -1)

for row in results.itertuples(index=False):
    x, y = sample[row.score_x], sample[row.score_y]
    if row.method == 'spearman':
        expected = spearmanr(x, y, nan_policy='raise').statistic
    else:
        expected = kendalltau(x, y, variant='b', nan_policy='raise').statistic
    assert math.isclose(row.coefficient, expected, rel_tol=0, abs_tol=1e-12)

print('Checks passed: known ties/reversed order, matrix symmetry, diagonals, bounds,')
print('663 occupations in every pair, and all 36 coefficients verified against raw-score calculations.')

Checks passed: known ties/reversed order, matrix symmetry, diagonals, bounds,
663 occupations in every pair, and all 36 coefficients verified against raw-score calculations.


## 3. Two headline beta heatmaps

One figure per Eloundou label source, with Spearman and Kendall panels. All panels share the −1 to +1 color scale and display numeric coefficients. Exact variants and the sample size are labeled. A negative correlation is retained as measured; it is not converted to positive agreement.

In [4]:
figures_folder.mkdir(exist_ok=True)
figure_paths = {}
for label_source, label in [('gpt4', 'GPT-4'), ('human', 'human')]:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.3), layout='constrained')
    labels = ['AIOE', 'Frey–Osborne', f'Eloundou\n{label} beta']
    for ax, method, title in zip(axes, methods, ['Spearman rho', 'Kendall tau-b']):
        matrix = matrices[('beta', label_source, method)]
        im = ax.imshow(matrix, cmap='RdBu', vmin=-1, vmax=1)
        ax.set_xticks(range(3), labels=labels)
        ax.set_yticks(range(3), labels=labels)
        ax.tick_params(axis='both', labelsize=10)
        ax.set_title(title, fontsize=13)
        for i in range(3):
            for j in range(3):
                value = matrix.iloc[i, j]
                ax.text(j, i, f'{value:.3f}', ha='center', va='center',
                        color='white' if abs(value) > 0.55 else 'black', fontsize=12)
    fig.suptitle(f'Occupation rank correlations — Eloundou {label} beta\n'
                 'Same 663 occupations in every pair', fontsize=15)
    fig.colorbar(im, ax=axes, label='Correlation coefficient', shrink=0.82)
    path = figures_folder / f'task10_{label_source}_beta_heatmaps.png'
    fig.savefig(path, dpi=180, facecolor='white')
    plt.show()
    plt.close(fig)
    figure_paths[label_source] = path
    print(f'Saved {path}')

Saved ../figures/task10_gpt4_beta_heatmaps.png
Saved ../figures/task10_human_beta_heatmaps.png


## 4. Export results and the sensitivity appendix

The appendix includes full alpha/gamma matrices for both label sources and both methods, beta-versus-alpha/gamma changes, and headline interpretation. Differences are descriptive; no significance test or confidence interval is claimed.

In [5]:
def markdown_table(frame, digits=4):
    # Avoid adding an optional table-formatting dependency.
    def cell(value):
        return f'{value:.{digits}f}' if isinstance(value, (float, np.floating)) else str(value)
    lines = ['| ' + ' | '.join(map(str, frame.columns)) + ' |',
             '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |']
    lines.extend('| ' + ' | '.join(cell(v) for v in row) + ' |'
                 for row in frame.itertuples(index=False, name=None))
    return '\n'.join(lines)

sensitivity = results.loc[~results['variant'].eq('beta')].copy()
beta_reference = results.loc[results['variant'].eq('beta'),
    ['label_source', 'method', 'index_x', 'index_y', 'coefficient']
].rename(columns={'coefficient': 'beta_coefficient'})
sensitivity = sensitivity.merge(
    beta_reference, on=['label_source', 'method', 'index_x', 'index_y'],
    validate='many_to_one',
)
sensitivity['change_from_beta'] = sensitivity['coefficient'] - sensitivity['beta_coefficient']
assert len(sensitivity) == 24 and sensitivity['beta_coefficient'].notna().all()

headline = paired.loc['beta'].reset_index()
headline_text = markdown_table(headline)
appendix_parts = [
    '# Task 10 — Rank-correlation results and sensitivity appendix',
    f'Input: `agreement_analysis_sample.csv`; SHA-256: `{source_sha256}`.',
    '## Methods',
    'All runs use exactly the same 663 unique six-digit 2018 SOC occupations from Task 9. '
    'Beta uses the saved percentile ranks. Alpha/gamma are ranked within these same occupations '
    'using ascending average ranks divided by 663. No imputation, employment weighting, '
    'or pair-specific row deletion is performed. Missing sensitivity scores stop execution.',
    'Spearman rho compares ranked positions; Kendall tau-b adjusts for ties. '
    'Each full matrix is symmetric with a diagonal of 1. Each pair has n = 663. '
    'The signed label difference is human minus GPT-4. A positive difference means '
    'the human coefficient is more positive; it does not necessarily mean stronger '
    'agreement when a coefficient is negative.',
    '## Headline beta coefficients',
    headline_text,
]
modern = paired.loc[('beta', 'spearman', 'aioe', 'eloundou')]
older = paired.loc[('beta', 'spearman', 'frey_osborne', 'eloundou')]
baseline_rho = paired.loc[('beta', 'spearman', 'aioe', 'frey_osborne'), 'gpt4']
appendix_parts.append(
    f'AIOE and Eloundou beta show a strong positive association: Spearman rho is '
    f'{modern["gpt4"]:.4f} for GPT-4 and {modern["human"]:.4f} for human labels. '
    f'The label-source change is small ({modern["human_minus_gpt4"]:+.4f}). '
    f'Frey–Osborne has a negative association with AIOE ({baseline_rho:.4f}) '
    f'and with Eloundou beta ({older["gpt4"]:.4f} for GPT-4; '
    f'{older["human"]:.4f} for human labels). Kendall tau-b gives the same '
    'directions in the headline runs.'
)
appendix_parts.extend([
    'Coefficients near +1 indicate similar ordering; near zero little consistent ordering; '
    'near −1 opposite ordering. Frey–Osborne estimates computerisation probability, while '
    'AIOE and Eloundou measure different forms of exposure. A negative relationship '
    'does not by itself establish a data error or show that either study is wrong.',
    '## Sensitivity coefficient changes',
    'All rows below also have n = 663. Change from beta is the alpha/gamma coefficient '
    'minus the beta coefficient for the same label source, method, and index pair.',
    markdown_table(sensitivity[
        ['variant', 'label_source', 'method', 'index_x', 'index_y', 'coefficient',
         'beta_coefficient', 'change_from_beta', 'human_minus_gpt4', 'n']
    ]),
])
for variant in ['alpha', 'gamma']:
    appendix_parts.append(f'## Full {variant} matrices')
    for label_source in label_sources:
        for method in methods:
            appendix_parts.append(f'### {label_source} {variant} — {method}, n = 663')
            matrix = matrices[(variant, label_source, method)]
            appendix_parts.append(markdown_table(matrix.rename_axis('index').reset_index()))

modern_changes = sensitivity.loc[sensitivity['index_y'].eq('eloundou'), 'change_from_beta']
largest_change = sensitivity.loc[sensitivity['change_from_beta'].abs().idxmax()]
alpha_rho = matrices[('alpha', 'gpt4', 'spearman')].loc['aioe', 'eloundou']
alpha_fo_rho = matrices[('alpha', 'gpt4', 'spearman')].loc['frey_osborne', 'eloundou']
appendix_parts.extend([
    '## Sensitivity summary',
    f'The largest absolute coefficient change from beta among pairs involving Eloundou '
    f'is {modern_changes.abs().max():.4f}: {largest_change.label_source} '
    f'{largest_change.variant}, {largest_change.method}, {largest_change.index_x} versus '
    f'{largest_change.index_y} changes from {largest_change.beta_coefficient:.4f} '
    f'to {largest_change.coefficient:.4f}. AIOE–Frey–Osborne is unchanged in all runs '
    'because neither score depends on the Eloundou label source or variant.',
    f'GPT-4 alpha gives a weaker AIOE association (Spearman {alpha_rho:.4f}) '
    f'than GPT-4 beta ({modern["gpt4"]:.4f}). GPT-4 alpha versus Frey–Osborne '
    f'is near zero ({alpha_fo_rho:.4f}), compared with the negative beta result '
    f'({older["gpt4"]:.4f}). The size, and in this case the sign, of a correlation '
    'can depend on the Eloundou measure. Alpha/gamma are alternative definitions, '
    'not new occupation samples; do not generalize the beta findings to every variant.',
    '## Outputs and reproduction',
    'Run `notebooks/task10.ipynb` from top to bottom from the repository root or '
    '`notebooks/`, using `requirements.txt`. Re-running replaces Task 10 outputs.',
    '- `data/datasets/processed/pairwise_correlations.csv`: 36 rows; three unique pairs '
    '× two methods × two label sources × three variants. Full precision is retained; '
    'the tables in this appendix are rounded to four decimals.',
    '- `figures/task10_gpt4_beta_heatmaps.png` and '
    '`figures/task10_human_beta_heatmaps.png`: two panels each, labeled Spearman/Kendall.',
    '- This appendix records full alpha/gamma matrices, label-source differences, '
    'and changes from the beta headline results. The full beta matrices are also '
    'displayed below in the notebook.',
    f'Validated with pandas {pd.__version__} and SciPy {scipy.__version__}.',
    '## Limits',
    'These are descriptive correlations, not significance tests or causal evidence. '
    'The analysis applies to the 663 occupations with complete index coverage. '
    'Original code-vintage, crosswalk, source-coverage, and measurement differences remain. '
    'Percentile ranks and exposure scores are not probabilities of job loss.',
])

assert sample_path.read_bytes() == input_bytes, 'Task 9 sample changed during this run'
results_path = processed_folder / 'pairwise_correlations.csv'
appendix_path = processed_folder / 'correlation_sensitivity_appendix.md'
results.to_csv(results_path, index=False)
appendix_path.write_text('\n\n'.join(appendix_parts) + '\n', encoding='utf-8')
saved = pd.read_csv(results_path)
pd.testing.assert_frame_equal(saved, results.reset_index(drop=True))
assert saved['n'].eq(663).all()
assert sample_path.read_bytes() == input_bytes

print(f'Saved {len(saved)} pairwise coefficient rows and the sensitivity appendix.')
print(f'Maximum absolute alpha/gamma change from beta: {modern_changes.abs().max():.4f}')
print('Full headline beta matrices:')
for label_source in label_sources:
    for method in methods:
        print(f'\n{label_source} beta — {method}')
        print(matrices[('beta', label_source, method)].to_string())
print('\nAll Task 10 checks passed, including CSV read-back and unchanged Task 9 input.')

Saved 36 pairwise coefficient rows and the sensitivity appendix.
Maximum absolute alpha/gamma change from beta: 0.4337
Full headline beta matrices:

gpt4 beta — spearman
                  aioe  frey_osborne  eloundou
aioe          1.000000     -0.367823  0.862805
frey_osborne -0.367823      1.000000 -0.267008
eloundou      0.862805     -0.267008  1.000000

gpt4 beta — kendall
                  aioe  frey_osborne  eloundou
aioe          1.000000     -0.249075  0.663246
frey_osborne -0.249075      1.000000 -0.182787
eloundou      0.663246     -0.182787  1.000000

human beta — spearman
                  aioe  frey_osborne  eloundou
aioe          1.000000     -0.367823  0.866948
frey_osborne -0.367823      1.000000 -0.340117
eloundou      0.866948     -0.340117  1.000000

human beta — kendall
                  aioe  frey_osborne  eloundou
aioe          1.000000     -0.249075  0.671174
frey_osborne -0.249075      1.000000 -0.233392
eloundou      0.671174     -0.233392  1.000000

All Task 10